# tensor-unbind — ex7: split heads for multi-head attention

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-unbind`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-unbind`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-unbind"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch unbind — quick refresher

`x.unbind(dim=k)` returns a tuple of `x.shape[k]` view-tensors with axis `k` removed. The result is a *Python tuple*, not a tensor — perfect for destructuring named components (`origin, direction = rays.unbind(dim=1)`) or for fanning a batched tensor into per-head / per-channel slices.

**Compared to `select`.** `unbind(dim=k)[i]` ≡ `select(k, i)`. Use `select` when you want ONE slice; use `unbind` when you want ALL of them. Both return views (no copy), so writes through the view alias the source.

### Exercise 7 — split heads for multi-head attention

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Apply `unbind(dim=2)` to fan a `(B, S, H, D)` attention tensor into a list of `(B, S, D)` per-head tensors, transform each, and restack while printing per-head norms for debug.
> Keywords: attention, multi-head, rearrange, per-head-debug
> ```

**KCs targeted:** `unbind-explicit-dim`, `unbind-tuple-destructure`

Implement `ex7_apply_per_head(x, scales)`. The canonical attention-head splitting pattern (without any matmul, so we focus on the unbind/restack mechanics):

1. `x` has shape `(B, S, H, D)` — batch, sequence, num_heads, head_dim.
2. Use `x.unbind(dim=2)` to get a length-`H` tuple of `(B, S, D)` per-head tensors.
3. For each head `h`, multiply by `scales[h]` (a scalar) and **print** `head_idx, scaled.norm()` so the caller can see the per-head magnitudes.
4. Restack with `t.stack(scaled_heads, dim=2)` to recover the `(B, S, H, D)` shape.

Inputs:
- `x`: `(B, S, H, D)` float tensor.
- `scales`: 1-D float tensor of length `H`.

Output: `(B, S, H, D)` float tensor where head `h` is scaled by `scales[h]`.

The visualization renders the per-head L2 norm bar chart from the real attention-shaped batch used in the smoke test.

In [ ]:
def ex7_apply_per_head(x: Tensor, scales: Tensor) -> Tensor:
    heads = x.unbind(dim=2)
    scaled = []
    for h, head in enumerate(heads):
        s = head * scales[h]
        print(f'  head {h}: norm={s.norm().item():.4f}')
        scaled.append(s)
    return t.stack(scaled, dim=2)


<details><summary>Solution</summary>

```python
def ex7_apply_per_head(x: Tensor, scales: Tensor) -> Tensor:
    heads = x.unbind(dim=2)
    scaled = []
    for h, head in enumerate(heads):
        s = head * scales[h]
        print(f'  head {h}: norm={s.norm().item():.4f}')
        scaled.append(s)
    return t.stack(scaled, dim=2)
```

**`unbind` + `stack` is the round-trip identity.** If you do `t.stack(x.unbind(dim=k), dim=k)`, you get `x` back. This is what lets per-head transforms compose: peel along the head axis, do anything you want with the per-head tensors, restack along the same axis.

**In real attention,** you wouldn't unbind heads — you'd just broadcast or use `einsum`. But the unbind/stack pattern is indispensable when each head needs a DIFFERENT module (e.g. per-head LoRA adapters, per-head dropout masks, mixture-of-experts gating).

**Why print per-head norms.** Dead heads (norm → 0) and saturating heads (norm → ∞) are the two failure modes of multi-head models. Logging per-head magnitudes during forward passes is the first-line diagnostic.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()